In [3]:
##### importing custom modules from the projects folder
import sys, os
from pathlib import Path
# Add project root to sys.path - search backwards through folders to find config.py
cwd = Path.cwd()
# Search upwards until a "config*" file is found
for parent in [cwd, *cwd.parents]:
    match = next(parent.glob('config*'), None)
    if match:
        PROJECT_ROOT = match.parent
        break
sys.path.append(str(PROJECT_ROOT))
import config
##### -------------------------------------------------
import scripts.functions.NBAhelperfunctions as hf

import json, time, requests, random
import pandas as pd
import numpy as np
from datetime import datetime
from bs4 import BeautifulSoup as bs
from sqlalchemy import create_engine

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.wait import WebDriverWait # used to wait for elements - popups
from selenium.webdriver.support import expected_conditions as EC  # used to wait for elements - popups
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select # used for drop down

today = datetime.today().date()
season = '2025'

In [ ]:
service = Service(config.BROWSER_DIR / 'geckodriver.exe')
driver = webdriver.Firefox(service=service)

In [ ]:
headers = {
    'Host': 'www.rotowire.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) Gecko/20100101 Firefox/144.0',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'DNT': '1',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Referer': 'https://www.rotowire.com/betting/nba/archive.php',
    'Cookie': 'PHPSESSID=1010a68a7a2ab43c750e881dc4f90c67; cookieyes-consent=consentid:Y20zNVltVERaZ2Z3ZExxaE8yMUVKU2dtUXFHeEZJS3U,consent:yes,action:no,necessary:yes,functional:yes,analytics:yes,performance:yes,advertisement:yes,other:yes; g_uuid=89bb25a1-9acc-45e4-ac16-c051aed33537; cohort_id=3; _ga_DJZM5GNYZ8=GS2.1.s1762610081$o51$g1$t1762610132$j22$l0$h1953729589; _ga=GA1.1.1911159050.1736898965; _au_1d=AU1D-0100-001736898967-KC8Y7NNE-Y6UV; euconsent=CQLOJoAQLOJoAGRABAENBYFgAAAAAAAAAAAAAAAVggAAAAAA.YAAAAAAAAAAA,; cto_bundle=qbIg-V9NN203aUdZJTJCNUx5aUsyRnJpVHc0azREJTJCMkJoWWg1bHozNUFTbnM3UmZqWEVCdHFteWhTcDVkUFJmc24lMkJnOUJZcG9HT1Q1Q3o0MFBqVFlyS09OZXdHenRJOTdBVmRPRkxDQzVRRHdHNnFYMkpET05XUlVkcGJFaWJHT0RpYVFzMEN5SW0xUzVMTFp5Y1dWVFVYYjRQV25UMDh0Y1ZYbyUyRlpNTmhKVjVOVk84cG9hWnNPeDVuWkMlMkIlMkZ4VnZzbGQlMkZiSg; cto_bidid=_3kuBF9OdDhPRVZYMHJ0Mm9IU1Y5NVVVY1FuRTRkOVpCdjlaJTJCZGM3aW91YkU1Y29Ic1N1Nk1xOHJhYnhIMkwlMkJOODJYbXJuNVRTRFVXNFpVMUZIOUhDSm1vZzVybHU3NUNNMHZkN2F0Z2xMeFhFaEklM0Q; _ga_FVWZ0RM4DH=GS2.1.s1761695340$o43$g0$t1761695340$j60$l0$h0; _lc2_fpi=ee48b0c2def8--01jjfeqwe5xs70dp1q6xdmcnzy; _lc2_fpi_meta=%7B%22w%22%3A1737833050565%7D; _cc_id=382d2e2c652690d3af967073410eb735; cto_dna_bundle=t9ehFF9JUURxMUQ2alJidm5yNFRwcFozU0E1UVhUJTJGRFlHNDNjVDJTSUVMelFGUyUyRjBHS0NSSm1mOHZNOXBkTkZBeDQxSGZ2TWhSVUNMZFZiTEJuZEpiWnMyNlElM0QlM0Q; _tt_enable_cookie=1; _ttp=Y7oNZ6GhF7I80if0fXvYesjlkDa.tt.1; 33acrossIdTp=Tg6wSz%2FBcdVJj2gKbImR6hB3Sebjl5qJpX1U5Fkbgws%3D; idw-fe-id=e90a95b4-e86d-4876-805c-ada63bcbcfff; __gads=ID=b282747500baff44:T=1737833053:RT=1761694233:S=ALNI_MaBArO9q9WZiiiL1FbYeDhvAOeHYA; __gpi=UID=00000fee44281a03:T=1737833053:RT=1761694233:S=ALNI_MYO3WueomqgLirGOjpXiboLQctcsQ; uuid=6E0EC57D-03FB-4CA6-8EFF-D00BA6D08015; sharedId=9f81a9e5-b52f-4174-88cc-e8e82cf860fc; sharedId_cst=zix7LPQsHA%3D%3D; _li_ss=CgA; connectId={"ttl":86400000,"lastUsed":1761694230914,"lastSynced":1761694230914}; ttcsid_CRLDHLJC77U51LO9QM5G=1744841529167.4.1744841529509; ttcsid=1744841529168.4.1744841529168; rw_tsd=1762610081__(direct)__(none)__(not set)__web__desktop; _gcl_au=1.1.2118381701.1761694229; _fbp=fb.1.1761694229103.775235795715729040; intercom-id-bfhjit7z=3580d51d-69d9-460f-a2ba-a59615f9dfc8; intercom-session-bfhjit7z=; intercom-device-id-bfhjit7z=c6d236b4-c3cd-49e9-a7aa-ea06df07e055; hb_insticator_uid=bcbb3954-8aa1-429f-8ef1-d8575736994b; __eoi=ID=b97afe3679878ef6:T=1761694233:RT=1761694233:S=AA-AfjYjPDAhTZU4npQaEV6JF2mr; _lr_env_src_ats=false; pbjs-unifiedid=%7B%22TDID%22%3A%22f5814713-9bbf-4568-897b-73224dbc34a3%22%2C%22TDID_LOOKUP%22%3A%22TRUE%22%2C%22TDID_CREATED_AT%22%3A%222025-09-28T23%3A30%3A39%22%7D; pbjs-unifiedid_cst=YiwPLDosoA%3D%3D; pbjs-unifiedid_last=Tue%2C%2028%20Oct%202025%2023%3A30%3A40%20GMT; _vwo_uuid_v2=DE174984BEC99CBF915E4F5C8FAAAC6D9|291ada161bdba5be2614e65ac5aaf624; _vwo_uuid=DE174984BEC99CBF915E4F5C8FAAAC6D9; _vwo_ds=3%3Aa_1%2Ct_1%3A0%241762275972%3A27.35666183%3A%3A%3A3_1%2C2_1%3A0%3A1762275972%3A1762275972; _vis_opt_s=1%7C; __stripe_mid=288c4141-3cfc-423b-a889-c107e6f0a0fe942431; g_device=windows%7Cdesktop; _vis_opt_test_cookie=1; rwlanding=%252Fbetting%252Fnba%252Farchive.php; g_sid=1762610080370.un5zhc9e; _rdt_uuid=1761694228917.08c5a2cc-d0c4-4641-8f17-151c9d049cd5; _uetsid=79d8ada0bcaa11f08a4219c9c3793b79; _uetvid=16bdcb20b45611f0872639b99b04fe34',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'TE': 'trailers'
}

url = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
r = requests.get(url, headers = headers)

data = json.loads(r.text)
df = pd.DataFrame(data)

AttributeError: 'Response' object has no attribute 'response'

In [18]:
df

,season,game_date,game_time,home_team_id,home_team_stats_id,home_team_abbrev,visit_team_id,visit_team_stats_id,visit_team_abbrev,home_team_score,...,start,favorite,score,total,spread,over_hit,under_hit,favorite_covered,underdog_covered,name
0,2017,2017-10-17 00:00:00,20,11,CLE,CLE,2,BOS,BOS,102,...,Night,CLE,99-102,201,4.5,0,1,0,1,"<span style=""color:#1da561 !important"">BOS</sp..."
1,2017,2017-10-17 00:00:00,22,22,GSW,GSW,17,HOU,HOU,121,...,Night,GSW,122-121,243,9.5,1,0,0,1,"<span style=""font-weight:700;color:#1da561 !im..."
2,2017,2017-10-18 00:00:00,19,12,DET,DET,30,CHA,CHA,102,...,Night,DET,90-102,192,2.5,0,1,1,0,"<span style="""">CHA</span> @ <span style=""font-..."
3,2017,2017-10-18 00:00:00,19,29,IND,IND,32,BKN,BKN,140,...,Night,IND,131-140,271,3.0,1,0,1,0,"<span style="""">BKN</span> @ <span style=""font-..."
4,2017,2017-10-18 00:00:00,19,5,ORL,ORL,1,MIA,MIA,116,...,Night,MIA,109-116,225,3.5,1,0,0,1,"<span style="""">MIA</span> @ <span style=""font-..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10242,2025,2025-11-07 00:00:00,20,1,MIA,MIA,30,CHA,CHA,126,...,Night,MIA,108-126,234,7.0,0,1,1,0,"<span style="""">CHA</span> @ <span style=""font-..."
10243,2025,2025-11-07 00:00:00,20,13,MIL,MIL,9,CHI,CHI,126,...,Night,MIL,110-126,236,4.5,0,1,1,0,"<span style="""">CHI</span> @ <span style=""font-..."
10244,2025,2025-11-07 00:00:00,20,18,MIN,MIN,20,UTA,UTA,137,...,Night,MIN,97-137,234,12.5,1,0,1,0,"<span style="""">UTA</span> @ <span style=""font-..."
10245,2025,2025-11-07 00:00:00,22,16,DEN,DEN,22,GSW,GSW,129,...,Night,DEN,104-129,233,9.5,1,0,1,0,"<span style="""">GSW</span> @ <span style=""font-..."


In [16]:
data[-1].keys()

dict_keys(['season', 'game_date', 'game_time', 'home_team_id', 'home_team_stats_id', 'home_team_abbrev', 'visit_team_id', 'visit_team_stats_id', 'visit_team_abbrev', 'home_team_score', 'visit_team_score', 'game_over_under', 'line', 'tipoff', 'month', 'start', 'favorite', 'score', 'total', 'spread', 'over_hit', 'under_hit', 'favorite_covered', 'underdog_covered', 'name'])

### betting pros nfl prop explore

In [ ]:

url = 'https://api.bettingpros.com/v3/offers'

# prop_name:[market id, number of entries at the time running. have to manually lookup on site]
# i can't figure out how to get a response > 5 players at a time so the total number is required
# to loop through pages
market_ids = {
    'total_rushing_yds':[301, 57],
    'total_rushing_tds':[305, 57],
    'total_receiving_yds':[302, 105],
    'total_receiving_tds':[306, 87],
    'total_passing_yds':[300, 33],
    'total_passing_tds':[304, 32]
}

scraped_json = {
    'total_rushing_yds':[],
    'total_rushing_tds':[],
    'total_receiving_yds':[],
    'total_receiving_tds':[],
    'total_passing_yds':[],
    'total_passing_tds':[]
}
book_id = None

params = {
    'sport': 'NFL',
    'market_id':None,    # 'marketId'
    'season': season,    # 'YYYY'
    'book_id': 'null',   
    'limit': '5',
    'page': '1'
}

headers = {
    'Host': 'api.bettingpros.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Referer': 'https://www.bettingpros.com/',
    'Origin': 'https://www.bettingpros.com',
    'DNT': '1',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site',
    'Priority': 'u=4',
    'TE': 'trailers',
    'x-api-key': 'CHi8Hy5CEE4khd46XNYL23dCFX96oUdw6qOt1Dnh'  # you’ll need to add this manually
}

for k,v in market_ids.items():
    
    # load market id for specific prop
    params['market_id'] = str(v[0])

    # have to loop through pages 5 players at a time
    n_players = v[1]
    n_pages = int(n_players / 5) + 1
    for i in range(1, n_pages + 1):

        params['page'] = i
        
        r = requests.get(
            url,
            headers=headers,
            params=params
        )
        print(k, i, r)
        try:
            #soup = bs(r.text, features='lxml')
            scraped_json[k].append(r.json())
            print('made soup')
        except:
            print('soup ruined')

        time.sleep(3)

total_rushing_yds 1 <Response [200]>
made soup
total_rushing_yds 2 <Response [200]>
made soup
total_rushing_yds 3 <Response [200]>
made soup
total_rushing_yds 4 <Response [200]>
made soup
total_rushing_yds 5 <Response [200]>
made soup
total_rushing_yds 6 <Response [200]>
made soup
total_rushing_yds 7 <Response [200]>
made soup
total_rushing_yds 8 <Response [200]>
made soup
total_rushing_yds 9 <Response [200]>
made soup
total_rushing_yds 10 <Response [200]>
made soup
total_rushing_yds 11 <Response [200]>
made soup
total_rushing_yds 12 <Response [200]>
made soup
total_rushing_tds 1 <Response [200]>
made soup
total_rushing_tds 2 <Response [200]>
made soup
total_rushing_tds 3 <Response [200]>
made soup
total_rushing_tds 4 <Response [200]>
made soup
total_rushing_tds 5 <Response [200]>
made soup
total_rushing_tds 6 <Response [200]>
made soup
total_rushing_tds 7 <Response [200]>
made soup
total_rushing_tds 8 <Response [200]>
made soup
total_rushing_tds 9 <Response [200]>
made soup
total_rush

In [ ]:
player_rows = []
for prop in scraped_json:
    # loop through scraped player pages - 5 players per page
    for i in scraped_json[prop]:

        for j in i['offers']:

            temp = j

            # all data for sinlge player
            # =========================
            individual_player_data = temp

            # meta data for player
            # =========================
            player_meta = individual_player_data['participants'][0]

            pid = player_meta['id']
            name = player_meta['name']
            pos = player_meta['player']['position']
            team = player_meta['player']['team']

            # line and odds data
            # =========================
            opening_line_data = individual_player_data['selections'][0]['opening_line']
            
            opening_line = opening_line_data['line']
            opening_odds = opening_line_data['cost']
            opening_bookid = opening_line_data['book_id']

            # book data 
            # =========================
            maps_bettingpros_books = {
                0:'consensus',
                13:'ceasars',
                10:'fanduel',
                37:'prizepicks',
                19:'betmgm',    
                33:'espnbet',
                27:'party casino',
                49:'hard rock'
            }
            for k in temp['selections'][0]['books']:

                book_id = k['id']
                if book_id != 0:
                    continue
                else:
                    line_data = k['lines'][0]
                    
                    current_odds = line_data['cost']
                    current_line = line_data['line']
                    isMain = line_data['main']
                    isBest = line_data['best']

            #####################
            player_rows.append([
                prop, today, pid, name, pos, team, 
                opening_line, opening_odds, opening_bookid,
                current_odds, current_line
            ])
    


headers = [
    'prop', 'date',
    'playerId', 'name', 'pos', 'team', 
    'opening_line', 'opening_odds', 'opening_bookid', 
    'current_odds', 'current_line'
]
df = pd.DataFrame(
    player_rows,
    columns=headers
)


In [96]:
for i in temp['offers'][0]['selections'][0]['books']:
    a = i['lines'][0]
    print(i['id'], a['cost'], a['line'],'\nmain:', a['main'], 'best:',a['best'])

0 -115 3750.5 
main: True best: False
10 -114 3750.5 
main: True best: False
37 -137 3699.5 
main: True best: False
19 -115 3650.5 
main: True best: False
33 -125 3500.5 
main: True best: True
13 -115 3750.5 
main: True best: False
27 -115 3650.5 
main: True best: False
49 -115 3700.5 
main: True best: False


### a